# VAR Model Notebook

This notebook covers:
1. **Setup** — Imports and helper functions
2. **Simulation** — Generating synthetic VAR(2) data and checking stationarity
3. **Model 1: VAR(2) on Synthetic Data** — Fitting `var_2.stan` to simulated data
4. **Model 2: Hierarchical VAR (Minnesota Prior) on Synthetic Data** — Fitting `var_2_hierarchical.stan`
5. **Model 3: VAR(2) on CTA Red Line Data** — Fitting `var_cta_test.stan` to real weekly ridership

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel, set_cmdstan_path

set_cmdstan_path("/deac/sta/classes/sta720/software/cmdstan/2.37.0")

In [ ]:
def sim_var_model(c, A_1, A_2, Sigma, T, seed=None):
    """Simulate a VAR(2) process.

    Parameters
    ----------
    c : array of shape (K,)
        Intercept vector.
    A_1, A_2 : arrays of shape (K, K)
        Lag-1 and lag-2 coefficient matrices.
    Sigma : array of shape (K, K)
        Innovation covariance matrix.
    T : int
        Number of time steps to simulate.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    Y : array of shape (T, K)
        Simulated time series.
    """
    np.random.seed(seed)
    assert T >= 1

    K = len(c)
    Y = np.zeros((T, K))

    epsilon = np.random.multivariate_normal(
        np.zeros(K), Sigma, size=T, check_valid='warn', tol=1e-8
    )

    mu = np.linalg.inv(np.eye(K) - A_1 - A_2) @ c
    Y[0] = np.random.multivariate_normal(mu, Sigma)
    Y[1] = np.random.multivariate_normal(mu, Sigma)

    for t in range(2, T):
        Y[t] = c + A_1 @ Y[t - 1] + A_2 @ Y[t - 2] + epsilon[t]
    return Y

In [ ]:
def lookup_station(value, stations_df):
    """Look up a CTA station by ID or name."""
    if isinstance(value, int):
        match = stations_df[stations_df["station_id"] == value]
        if not match.empty:
            return match.iloc[0]["stationname"]
    else:
        match = stations_df[stations_df["stationname"] == value]
        if not match.empty:
            return match.iloc[0]["station_id"]
    return None

---
## 2. Simulate Synthetic VAR(2) Data

In [ ]:
# True parameters for the synthetic VAR(2)
T_sim = 200
c_true = np.array([2.5, 1.5])
A_1_true = np.array([[0.45, -0.35],
                     [-0.15, 0.30]])
A_2_true = np.array([[0.05, -0.05],
                     [-0.05, 0.03]])
Sigma_true = np.array([[1.0, 0.1],
                       [0.1, 0.5]])

### 2.1 Stationarity Check

A VAR(2) is stable if all eigenvalues of the companion matrix have modulus < 1.

In [ ]:
companion = np.block([
    [A_1_true, A_2_true],
    [np.eye(2), np.zeros((2, 2))]
])
eigenvalues = np.linalg.eigvals(companion)
print("Companion matrix eigenvalue moduli:", np.abs(eigenvalues))
assert np.all(np.abs(eigenvalues) < 1), "VAR(2) is not stationary!"
print("All eigenvalues have modulus < 1 — process is stationary.")

### 2.2 Generate and Plot Synthetic Series

In [ ]:
Y_sim = sim_var_model(c_true, A_1_true, A_2_true, Sigma_true, T_sim, seed=None)

df_sim = pd.DataFrame(Y_sim, columns=["y1", "y2"])
df_sim["t"] = range(T_sim)
df_sim_long = df_sim.melt(id_vars="t", var_name="series", value_name="value")

plt.figure(figsize=(10, 4))
sns.lineplot(data=df_sim_long, x="t", y="value", hue="series")
plt.title("Synthetic VAR(2) Simulation")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

---
## 3. Model 1: VAR(2) on Synthetic Data

Fitting `var_2.stan` — a basic VAR(2) with LKJ correlation prior.

In [ ]:
stan_data_sim = {
    "T": T_sim,
    "K": len(c_true),
    "Y": Y_sim
}

var_mod = CmdStanModel(stan_file="var_2.stan")
var_mod_post = var_mod.sample(data=stan_data_sim)
print(var_mod_post.diagnose())

### 3.1 Parameter Recovery

In [ ]:
var_mod_params = var_mod_post.stan_variables()

print("True c:", c_true)
print("Estimated c:", var_mod_params["c"].mean(axis=0))
print()
print("True A_1:\n", A_1_true)
print("Estimated A_1:\n", var_mod_params["A_1"].mean(axis=0))
print()
print("True A_2:\n", A_2_true)
print("Estimated A_2:\n", var_mod_params["A_2"].mean(axis=0))

In [ ]:
print(var_mod_post.summary())

---
## 4. Model 2: Hierarchical VAR (Minnesota Prior) on Synthetic Data

Fitting `var_2_hierarchical.stan` — a VAR(P) with Minnesota-style shrinkage prior
controlled by hyperparameters `lambda` (overall tightness) and `theta` (cross-equation
relative tightness).

In [ ]:
stan_data_hier = {
    "T": T_sim,
    "K": len(c_true),
    "P": 2,
    "Y": Y_sim
}

var_mod_hier = CmdStanModel(stan_file="var_2_hierarchical.stan")
var_mod_hier_post = var_mod_hier.sample(data=stan_data_hier)
print(var_mod_hier_post.diagnose())

### 4.1 Parameter Recovery

In [ ]:
var_mod_hier_params = var_mod_hier_post.stan_variables()

print("True c:", c_true)
print("Estimated c:", var_mod_hier_params["c"].mean(axis=0))
print()
print("True A_1:\n", A_1_true)
print("True A_2:\n", A_2_true)
print("Estimated A (lag 1, lag 2):\n", var_mod_hier_params["A"].mean(axis=0))

In [ ]:
print(var_mod_hier_post.summary())

---
## 5. Model 3: VAR(2) on CTA Red Line Data

Applying the basic VAR(2) to real weekly differenced ridership from 4 Red Line stations.

### 5.1 Load and Inspect Data

In [ ]:
stations = pd.read_csv("../../data/clean/station-lookup.csv")

# Selected Red Line stations
selected_station_ids = [40080, 40100, 40190, 40240]
selected_station_cols = [f"s_{sid}" for sid in selected_station_ids]

print("Selected stations:")
for sid in selected_station_ids:
    name = lookup_station(sid, stations)
    print(f"  {sid}: {name}")

In [ ]:
cta_data_red_weekly = pd.read_csv("../../data/clean/cta-data-red-weekly.csv")

print(f"Full dataset shape: {cta_data_red_weekly.shape}")
print(f"Columns: {list(cta_data_red_weekly.columns[:5])} ...")
print(f"Date range: {cta_data_red_weekly['week'].min()} to {cta_data_red_weekly['week'].max()}")

In [ ]:
# Extract selected stations and drop the week column
cta_selected = cta_data_red_weekly[selected_station_cols].to_numpy()

print(f"Selected data shape: {cta_selected.shape}")
print(f"  T (weeks): {cta_selected.shape[0]}")
print(f"  K (stations): {cta_selected.shape[1]}")
print(f"  NaN count: {np.isnan(cta_selected).sum()}")

### 5.2 Fit VAR(2)

In [ ]:
stan_data_cta = {
    "T": cta_selected.shape[0],
    "K": cta_selected.shape[1],
    "Y": cta_selected
}

var_cta_mod = CmdStanModel(stan_file="var_cta_test.stan")
var_cta_post = var_cta_mod.sample(data=stan_data_cta)
print(var_cta_post.diagnose())

### 5.3 Results

In [ ]:
var_cta_params = var_cta_post.stan_variables()
print(var_cta_post.summary())